# Problema canibalilor și misionarilor

Pe malul râului se află N canibali și N misionari. Lângă ei se află o barcă cu maxim. M locuri. Toată lumea vrea să ajungă pe partea cealaltă a râului, dar nimeni nu știe să înoate.

Nici pe barcă, nici pe vreun mal nu trebuie să fie vreodată mai mulți canibali decât misionari (altfel o să îi mănânce). În barcă trebuie să fie mereu minim un om și maxim M oameni. Permutările între mal și barcă au loc instantaneu, așadar nu ne punem problema ordinii în care coboară sau urcă oamenii în barcă.

Care este o secvență de acțiuni în urma căreia toți oamenii să ajungă pe celălalt mal fără ca misionarii să ajungă prânz pentru canibali?

*Problema clasica este cu N=3 (canibali / misionari) și M=2 (locuri în barcă).*

## 1. Clasa Nod

Modificați informația clasei Nod pentru datele jocului curent. Noua informație va include:
* numărul de canibali pe malul stâng;
* numărul de misionari pe malul stâng;
* poziția bărcii.

Citiți dintr-un **_fișier_** valorile parametrilor N și M și păstrați-le ca elemente statice în clasă. Puteți pune valori default în clasă N=3 și M=2.

Implementați următoarele metode:
* constructorul clasei -> *\_\_init__*
* egalitatea -> *\_\_eq__*
* transformarea în obiect de tip string a informației nodului curent -> *\_\_str__*
* transformarea în obiect de tip string a informației elementelor dintr-o listă -> *\_\_repr__*

_Observații:_
* Informația nodului curent va fi afișată de forma:
```
Stare curentă:
3 misionari, 3 canibali  | 0 misionari, 0 canibali
Barca se află pe malul stâng
```
* Informația elementului dintr-o listă va fi afișată de forma:
```
(3, 3, 0)
```
Unde tuplul reprezintă (<număr misionari pe malul stâng>, <număr canibali pe malul stâng>, <malul pe care se află barca>)

In [ ]:
from __future__ import annotations

class Nod:
    N = 3 # oameni
    M = 2 # locuri in barca

    def __init__(self, m : int, c : int, p : int, parinte : Nod | None = None):
        self.parinte = parinte
        self.m = m
        self.c = c
        self.p = p

    def __eq__(self, other : Nod) -> bool:
        return (self.m == other.m and self.c == other.c and self.p == other.p)

    def __str__(self) -> str:
        result = "Stare curenta:\n"
        result += f"{self.m} misionari, {self.c} canibali | {self.N - self.m} misionari, {self.N - self.c} canibali\n"
        if self.p == 0:
            result += "Barca se afla pe malul stang\n"
        else:
            result += "Barca se afla pe malul drept\n"
        return result

    def __repr__(self) -> str:
        return f"({self.m}, {self.c}, {self.p})"

    def drumRadacina(self) -> list[Nod]:
        nod_curent = self
        drum = []
        while nod_curent is not None:
            drum.insert(0, nod_curent)
            nod_curent = nod_curent.parinte
        return drum

    def valid(self) -> bool:
        if not (0 <= self.m <= self.N and 0 <= self.c <= self.N):
            return False

        if self.m > 0 and self.c > self.m:
            return False

        m_dreapta = self.N - self.m
        c_dreapta = self.N - self.c
        if m_dreapta > 0 and m_dreapta < c_dreapta:
            return False

        return True



In [64]:
n = Nod(2, 3, 1)
l = [n, Nod(2, 3, 0)]
print(n)
print(l)
print(n.valid())

Stare curenta:
2 misionari, 3 canibali | 1 misionari, 0 canibali
Barca se afla pe malul drept
[(2, 3, 1), (2, 3, 0)]
False


## 2. Clasa Graf

Modificați următoarele funcții din clasa Graf pentru problema curentă:
* scop - verifică dacă toți oamenii au ajuns pe malul drept
* succesori - generează lista de succesori valizi ai stării curente

_Sugestie:_ Pentru a scrie mai puțin, puteți calcula succesorii unei stări în clasa Nod și apela acea funcție din clasa Graf. Nu uitați să verificați că succesorul nu a mai fost vizitat în drumul de la rădăcină!

In [ ]:
class Graf:
    def __init__(self, root: Nod):
        self.root = root

    def scop(self, nod : Nod) -> bool:
        return (nod.m == 0 and nod.c == 0 and nod.p == 1)

    def succesori(self, nod : Nod) -> list[Nod]:
        lista_succesori = []
        drum_curent = nod.drumRadacina()

        for b_m in range(Nod.N + 1):
            for b_c in range(Nod.N + 1):
                if 1 <= b_m + b_c <= Nod.M:

                    # barca ca -1 / 1
                    if nod.p == 0:
                        m_nou = nod.m - b_m
                        c_nou = nod.c - b_c
                        p_nou = 1

                    else:
                        m_nou = nod.m + b_m
                        c_nou = nod.c + b_c
                        p_nou = 0

                    nod_nou = Nod(m_nou, c_nou, p_nou, parinte=nod)

                    if nod_nou.valid() and nod_nou not in drum_curent:
                        lista_succesori.append(nod_nou)
        return lista_succesori

In [65]:
g = Graf(Nod(3, 3, 0))

g.succesori(Nod(3, 3, 0))

[(3, 2, 1), (3, 1, 1), (2, 2, 1)]

In [49]:
from collections import deque

def bfs(graf : Graf, n : int = 1) -> list[Nod]:
    radacina = graf.root
    q = deque([radacina])

    result = []

    while n > 0 and len(q) > 0:
        nod_curent = q.popleft()

        if graf.scop(nod_curent):
            result.append(nod_curent)
            n -= 1
            continue

        succesori = graf.succesori(nod_curent)
        q.extend(succesori)

    return result

In [51]:
bfs(g, 2)

[(0, 0, 1), (0, 0, 1)]

## 3. Căutarea drumului

Creați o funcție *printDrumRadacina()* care returnează drumul până la nodul curent în formatul de mai jos. Rulați problema cu 2 metode de căutare neinformată pentru N=3, M=2 și NSOL=2 (numărul de soluții cerute) și afișați într-un fișier cu calea dată de la tastatură drumul de la rădăcină până la nodul curent (în formatul precizat anterior) și timpul de rulare al algoritmului:

```
Stare curentă:
3 misionari, 3 canibali  | 0 misionari, 0 canibali
Barca se află pe malul stâng

Barca s-a deplasat de pe malul stâng pe malul drept cu 0 misionari și 2 canibali.

Stare curentă:
3 misionari, 1 canibali  | 0 misionari, 2 canibali
Barca se află pe malul drept

...

Stare curentă:
0 misionari, 0 canibali  | 3 misionari, 3 canibali
Barca se află pe malul drept

Timpul de rulare: 0.00021839141845703125 secunde.
```

In [67]:
from time import perf_counter

def printDrumRadacina(nsol : int = 2) -> None:
    g = Graf(Nod(3, 3, 0))
    start = perf_counter()
    sols = bfs(g, nsol)
    time_spent = perf_counter() - start
    for i in range(nsol):
        print("-"* 30)
        print(f"Path {i + 1}".center(30))
        print("-"* 30)

        drum = sols[i].drumRadacina()

        for cur_node in drum:
            print(str(cur_node) + "\n")

        print (f"Time {time_spent}")

printDrumRadacina()

------------------------------
            Path 1            
------------------------------
Stare curenta:
3 misionari, 3 canibali | 0 misionari, 0 canibali
Barca se afla pe malul stang

Stare curenta:
3 misionari, 1 canibali | 0 misionari, 2 canibali
Barca se afla pe malul drept

Stare curenta:
3 misionari, 2 canibali | 0 misionari, 1 canibali
Barca se afla pe malul stang

Stare curenta:
3 misionari, 0 canibali | 0 misionari, 3 canibali
Barca se afla pe malul drept

Stare curenta:
3 misionari, 1 canibali | 0 misionari, 2 canibali
Barca se afla pe malul stang

Stare curenta:
1 misionari, 1 canibali | 2 misionari, 2 canibali
Barca se afla pe malul drept

Stare curenta:
2 misionari, 2 canibali | 1 misionari, 1 canibali
Barca se afla pe malul stang

Stare curenta:
0 misionari, 2 canibali | 3 misionari, 1 canibali
Barca se afla pe malul drept

Stare curenta:
0 misionari, 3 canibali | 3 misionari, 0 canibali
Barca se afla pe malul stang

Stare curenta:
0 misionari, 1 canibali | 3 misionari